## I. Programowanie Liniowe (LP)

### Zadanie 1: Optymalizacja asortymentu w stolarni
**Treść:**  
Zakład produkuje biurka  i regały . Zysk ze sprzedaży biurka wynosi 300 zł, a regału 200 zł. Do produkcji wykorzystuje się drewno oraz czas pracy montażystów:
* Biurko wymaga 2 m² drewna i 4 godzin montażu.
* Regał wymaga 3 m² drewna i 2 godzin montażu.

Zakład dysponuje 60 m² drewna oraz 80 godzinami pracy. Popyt na biurka wynosi co najmniej 5 sztuk. Wyznacz plan produkcji maksymalizujący zysk.


In [5]:
# x0 - biurka
# x1 - regały

import numpy as np
from scipy.optimize import linprog
# Funkcja celu
c = [-300, -200]
# Macierz lewej strony
A_ub = [[2, 3], [4, 2]]
# Macierz prawej strony
b_ub = [60, 80]

bounds = [(5, None), (0, None)]
integrality = 1

result = linprog(c, A_ub = A_ub, b_ub = b_ub, bounds=bounds, method="highs", integrality = integrality, options={"disp":False})

if result.success:
    print(f"Optima: x0={result.x[0]}, x1={result.x[1]}, maksymalizacja celu c={-result.fun}")
else:
    print("Bład")

Optima: x0=14.999999999999998, x1=10.000000000000002, maksymalizacja celu c=6500.0


### Zadanie 2: Minimalizacja kosztu paszy dla zwierząt
**Treść:**  
Hodowca przygotowuje mieszankę paszową z dwóch składników: kukurydzy  i śruty sojowej. 1 kg kukurydzy kosztuje 1.20 zł, a 1 kg soi 2.50 zł. Mieszanka musi dostarczyć:
* Co najmniej 180 g białka (kukurydza zawiera 90 g/kg, soja 450 g/kg).
* Co najmniej 3000 kcal energii (kukurydza dostarcza 3300 kcal/kg, soja 2200 kcal/kg).
Łączna masa mieszanki nie może przekroczyć 3 kg. Dobierz skład mieszanki o minimalnym koszcie.

In [6]:
import numpy as np
from scipy.optimize import linprog

c = np.array([1.2, 2.5])
A_ub=np.array([
    [-90, -450],
    [-3300, -2200],
    [1, 1]
])

b_ub = np.array([-180, -3000, 3])

bounds = [(0, None), (0, None)]

result = linprog(c, A_ub = A_ub, b_ub = b_ub, bounds=bounds, method="highs", integrality = integrality, options={"disp":False})

if result.success:
    print(f"Optima: x0={result.x[0]}, x1={result.x[1]}, maksymalizacja celu c={result.fun}")
else:
    print("Bład")


Optima: x0=2.0, x1=-0.0, maksymalizacja celu c=2.4


### Zadanie 3: Kampania marketingowa online

Agencja planuje budżet na reklamy w wyszukiwarce oraz w mediach społecznościowych. Koszt jednego kliknięcia (CPC) wynosi 2 zł w wyszukiwarce i 1 zł w social mediach. 

* Całkowity budżet wynosi maksymalnie 10 000 zł.
* Ze względów wizerunkowych liczba kliknięć z mediów społecznościowych nie może być mniejsza niż z wyszukiwarki.
* Łączna liczba kliknięć musi wynieść co najmniej 6 000.

Każde kliknięcie w wyszukiwarce przynosi średnio 0.05 konwersji (zakupu), a w mediach społecznościowych 0.02 konwersji. 

**Polecenie:**  
Sformułuj zadanie programowania liniowego i znajdź taką liczbę kliknięć z obu kanałów, która pozwoli zmaksymalizować łączną liczbę konwersji.

In [7]:
import numpy as np
from scipy.optimize import linprog

c = np.array([-0.05, -0.02])
A_ub=np.array([
    [2, 1],
    [-1, 1],
    [-1, -1]
])

b_ub = np.array([10000, 0, 6000])

bounds = [(0, None), (0, None)]

result = linprog(c, A_ub = A_ub, b_ub = b_ub, bounds=bounds, method="highs", integrality = integrality, options={"disp":False})

if result.success:
    print(f"Optima: x0={result.x[0]}, x1={result.x[1]}, maksymalizacja celu c={-result.fun}")
else:
    print("Bład")

Optima: x0=5000.0, x1=0.0, maksymalizacja celu c=250.0


### Zadanie 4: Priorytetowe zarządzanie fabryką (Leksykograficzne GP)

Zakład wytwarza dwa wyroby: Produkt A oraz Produkt B. Czas produkcji wynosi odpowiednio 2 godziny dla Produktu A i 3 godziny dla Produktu B. Łączny dostępny czas pracy maszyn wynosi 120 roboczogodzin (jest to ograniczenie sztywne). 

Kierownictwo ustaliło trzy cele w ścisłej hierarchii priorytetów ($P_1 \gg P_2 \gg P_3$):
1. **Priorytet 1 :** Osiągnąć zysk na poziomie co najmniej 2 400 zł (zysk ze sprzedaży Produktu A to 40 zł/szt., a Produktu B to 60 zł/szt.). Minimalizujemy niedobór zysku.
2. **Priorytet 2 :** Wyprodukować dokładnie 20 sztuk Produktu A. Minimalizujemy wszelkie odchylenia od tej wielkości (zarówno nadmiar, jak i niedobór).
3. **Priorytet 3 :** Ograniczyć łączne zużycie surowca do maksymalnie 100 kg (Produkt A zużywa 2 kg surowca na sztukę, Produkt B zużywa 3 kg/szt.). Minimalizujemy przekroczenie limitu.

**Polecenie:**  
Zbuduj model leksykograficznego programowania celowego (Goal Programming) i wyznacz optymalny plan produkcji.

In [10]:
import numpy as np
from scipy.optimize import linprog

# [a, b, d1+, d1-, d2+, d2-, ..., dn+, dn-]
# Minimalizujemy tylko niektóre odchyłki zgodnie z priorytetami polecenia (hierarchia)
# Funkcja celu
c = [0, 0, 0, 1, 1, 1, 1, 0]
# Lewa strona równań wraz z odchyłkami
A_eq = [[40, 60, -1, 1, 0, 0, 0, 0],
        [1, 0, 0, 0, -1, 1, 0, 0],
        [2, 3, 0, 0, 0, 0, -1, 1]
        ]
# Prawa strona równań
b_eq = [2400, 20, 100]

# Górne ograniczenia nie do przkroczenia
A_ub = [[2, 3, 0, 0, 0, 0, 0, 0]]
b_ub = [120]
bounds = [(0, None)] * 8

result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs', integrality=1)


if result.success:
    x1, x2, d1_minus, d1_plus, d2_minus, d2_plus, d3_minus, d3_plus = result.x
    
    print("--- Wyniki Optymalizacji ---")
    print(f"Status: {result.message}")
    print("\n--- Wartości Zmiennych Decyzyjnych ---")
    print(f"Produkt A (x1) = {x1:.2f} szt.")
    print(f"Produkt B (x2) = {x2:.2f} szt.")
    
    print("\n--- Zmienne Odchyleń ---")
    print(f"Cel I   (Zysk):     niedomiar (d1-) = {d1_minus:.2f}, nadwyżka (d1+) = {d1_plus:.2f}")
    print(f"Cel II  (Wolumen):  niedomiar (d2-) = {d2_minus:.2f}, nadwyżka (d2+) = {d2_plus:.2f}")
    print(f"Cel III (Surowiec): niedomiar (d3-) = {d3_minus:.2f}, nadwyżka (d3+) = {d3_plus:.2f}")

    print("\n--- Osiągnięcie Celów ---")
    zysk = 40 * x1 + 60 * x2
    wolumen_A = x1
    surowiec = 2 * x1 + 3 * x2
    maszyny = 2 * x1 + 3 * x2
    
    print(f"Cel I (Zysk >= 2400 zł):      Osiągnięto = {zysk:.2f} zł")
    print(f"Cel II (Produkt A == 20):     Osiągnięto = {wolumen_A:.2f} szt.")
    print(f"Cel III (Surowiec <= 100 kg): Osiągnięto = {surowiec:.2f} kg")
    print(f"Zużycie czasu maszyn (<=120):              = {maszyny:.2f} h")
else:
    print("Nie udało się znaleźć rozwiązania.")
    print(f"Status: {result.message}")


--- Wyniki Optymalizacji ---
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)

--- Wartości Zmiennych Decyzyjnych ---
Produkt A (x1) = 21.00 szt.
Produkt B (x2) = 26.00 szt.

--- Zmienne Odchyleń ---
Cel I   (Zysk):     niedomiar (d1-) = 0.00, nadwyżka (d1+) = 0.00
Cel II  (Wolumen):  niedomiar (d2-) = 1.00, nadwyżka (d2+) = 0.00
Cel III (Surowiec): niedomiar (d3-) = 20.00, nadwyżka (d3+) = 0.00

--- Osiągnięcie Celów ---
Cel I (Zysk >= 2400 zł):      Osiągnięto = 2400.00 zł
Cel II (Produkt A == 20):     Osiągnięto = 21.00 szt.
Cel III (Surowiec <= 100 kg): Osiągnięto = 120.00 kg
Zużycie czasu maszyn (<=120):              = 120.00 h


### Zadanie 5: Ważone planowanie floty kurierskiej (Ważone GP)

Firma kurierska planuje zakup nowych pojazdów: małych aut dostawczych w cenie 100 tys. zł za sztukę oraz dużych aut dostawczych w cenie 180 tys. zł za sztukę. Z powodów operacyjnych flota musi powiększyć się o łącznie co najmniej 8 pojazdów (ograniczenie sztywne).

Zarząd zdefiniował dwa cele cząstkowe i przypisał im wagi ważności:
1. **Cel 1 (waga = 3):** Zmieścić się w łącznym budżecie 1 500 tys. zł. Minimalizujemy przekroczenie budżetu.
2. **Cel 2 (waga = 1):** Łączna ładowność nowych pojazdów powinna wynieść co najmniej 40 ton (ładowność małego auta to 2 tony, a dużego to 5 ton). Minimalizujemy niedobór łącznej ładowności względem celu.

**Polecenie:**  
Sformułuj model ważonego programowania celowego (Archimedesowego) i wyznacz optymalną liczbę kupowanych aut każdego typu.

In [ ]:
import numpy as np
from scipy.optimize import linprog

# [x1, x2, d1_minus, d1_plus, d2_minus, d2_plus]
c = [0, 0, 0, 1, 1, 0]

A_eq = [
    [100000, 180000, 1, -1, 0, 0],
    [2, 5, 0, 0, 1, -1]
]

b_eq = [1500000, 40]

A_ub = [[-1, -1, 0, 0, 0, 0]]

b_ub = [-8]

bounds = [(0, None)]*6

result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs', integrality=1)

if result.success:
    # Kolejność: [x1, x2, d1_minus, d1_plus, d2_minus, d2_plus]
    x1, x2, d1_minus, d1_plus, d2_minus, d2_plus = result.x
    
    print("=" * 45)
    print("        WYNIKI OPTYMALIZACJI (ZADANIE 5)")
    print("=" * 45)
    print(f"Status: {result.message}")
    print(f"Wartość ważonej funkcji kary: {result.fun:.2f}")

    print("\n--- Plan Zakupu Floty Pojazdów ---")
    print(f"Małe auta dostawcze (x1): {x1:.2f} szt.")
    print(f"Duże auta dostawcze (x2): {x2:.2f} szt.")
    print(f"Łączna liczba zakupionych aut: {x1 + x2:.2f} szt. (wymóg: >= 8)")

    print("\n--- Analiza Odchyleń od Celów ---")
    print(f"Cel I  (Budżet 1500 tys. zł):")
    print(f"  - Oszczędność (d1-):       {d1_minus:.2f} tys. zł")
    print(f"  - Przekroczenie (d1+):     {d1_plus:.2f} tys. zł (karane z wagą 3)")
    
    print(f"Cel II (Ładowność 40 ton):")
    print(f"  - Niedobór ładowności (d2-): {d2_minus:.2f} t (karane z wagą 1)")
    print(f"  - Nadwyżka ładowności (d2+): {d2_plus:.2f} t")

    print("\n--- Weryfikacja Realizacji Celów ---")
    koszt_calkowity = 100000 * x1 + 180000 * x2
    ladownosc_calkowita = 2 * x1 + 5 * x2
    
    print(f"Wydany budżet:     {koszt_calkowity:.2f} tys. zł / 1500 tys. zł")
    print(f"Łączna ładowność:  {ladownosc_calkowita:.2f} ton / 40 ton")
    print("=" * 45)
else:
    print("Nie udało się znaleźć optymalnego rozwiązania.")
    print(f"Status: {result.message}")

        WYNIKI OPTYMALIZACJI (ZADANIE 5)
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Wartość ważonej funkcji kary: 0.00

--- Plan Zakupu Floty Pojazdów ---
Małe auta dostawcze (x1): 0.00 szt.
Duże auta dostawcze (x2): 8.00 szt.
Łączna liczba zakupionych aut: 8.00 szt. (wymóg: >= 8)

--- Analiza Odchyleń od Celów ---
Cel I  (Budżet 1500 tys. zł):
  - Oszczędność (d1-):       60000.00 tys. zł
  - Przekroczenie (d1+):     0.00 tys. zł (karane z wagą 3)
Cel II (Ładowność 40 ton):
  - Niedobór ładowności (d2-): 0.00 t (karane z wagą 1)
  - Nadwyżka ładowności (d2+): 0.00 t

--- Weryfikacja Realizacji Celów ---
Wydany budżet:     1440000.00 tys. zł / 1500 tys. zł
Łączna ładowność:  40.00 ton / 40 ton


### Zadanie 6: Grafik dyżurów personelu medycznego (Ważone GP)

Szpital planuje liczbę dyżurów dziennych oraz nocnych w nadchodzącym miesiącu. Koszt obsadzenia dyżuru dziennego wynosi 400 zł, a dyżuru nocnego 700 zł. Dyrekcja wyznaczyła trzy cele o zróżnicowanych wagach:

* **Cel 1 (waga = 5):** Zapewnić łącznie dokładnie 90 dyżurów w miesiącu. Niepożądany jest zarówno brak pełnej obsady, jak i nadmiar pracowników (minimalizujemy sumę odchyleń w obie strony).
* **Cel 2 (waga = 2):** Utrzymać łączne wydatki na poziomie 45 000 zł. Minimalizujemy przekroczenie tego budżetu.
* **Cel 3 (waga = 1):** Zrealizować co najmniej 30 dyżurów nocnych. Minimalizujemy ewentualny niedobór dyżurów nocnych.

**Polecenie:**  
Zbuduj model programowania celowego z funkcją kary i wyznacz planowaną liczbę dyżurów dziennych oraz nocnych.

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Kolejność zmiennych:
# [x1, x2, d1_minus, d1_plus, d2_minus, d2_plus, d3_minus, d3_plus]
# x1: dyżury dzienne, x2: dyżury nocne

# Wagi kar w funkcji celu:
# Cel 1 (waga 5): d1_minus + d1_plus
# Cel 2 (waga 2): d2_plus (przekroczenie budżetu)
# Cel 3 (waga 1): d3_minus (deficyt nocy)
c = [0, 0, 5, 5, 0, 2, 1, 0]

# Ograniczenia celowe (wszystkie warunki są celami, brak sztywnych A_ub)
A_eq = np.array([
    [  1,   1,  1, -1,  0,  0,  0,  0],  # x1 + x2 + d1- - d1+ = 90
    [400, 700,  0,  0,  1, -1,  0,  0],  # 400*x1 + 700*x2 + d2- - d2+ = 45 000
    [  0,   1,  0,  0,  0,  0,  1, -1]   # x2 + d3- - d3+ = 30
])

b_eq = np.array([90, 45000, 30])

bounds = [(0, None)] * 8

# Dyżury personelu są liczbami całkowitymi
integrality = [1, 1, 0, 0, 0, 0, 0, 0]

result = linprog(
    c, 
    A_eq=A_eq, 
    b_eq=b_eq, 
    bounds=bounds, 
    method="highs", 
    integrality=integrality
)

if result.success:
    x1, x2, d1_m, d1_p, d2_m, d2_p, d3_m, d3_p = result.x
    
    print("=" * 50)
    print("        WYNIKI OPTYMALIZACJI (ZADANIE 6)")
    print("=" * 50)
    print(f"Status: {result.message}")
    print(f"Wartość łącznej funkcji kary: {result.fun:.2f}")

    print("\n--- Plan Dyżurów ---")
    print(f"Dyżury dzienne (x1): {x1:.0f}")
    print(f"Dyżury nocne   (x2): {x2:.0f}")
    print(f"Łącznie dyżurów:    {x1 + x2:.0f}")

    print("\n--- Analiza Odchyleń ---")
    print(f"Cel I  (Liczba dyżurów = 90):     niedobór = {d1_m:.0f}, nadmiar = {d1_p:.0f}")
    print(f"Cel II (Budżet = 45 000 zł):      oszczędność = {d2_m:.2f} zł, przekroczenie = {d2_p:.2f} zł")
    print(f"Cel III (Dyżury nocne >= 30):     niedobór = {d3_m:.0f}, nadmiar = {d3_p:.0f}")

    print("\n--- Podsumowanie Kosztów ---")
    koszt = 400 * x1 + 700 * x2
    print(f"Rzeczywisty koszt obsady: {koszt:.2f} zł")
    print("=" * 50)
else:
    print(f"Błąd optymalizacji: {result.message}")

        WYNIKI OPTYMALIZACJI (ZADANIE 6)
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Wartość łącznej funkcji kary: 0.00

--- Plan Dyżurów ---
Dyżury dzienne (x1): 60
Dyżury nocne   (x2): 30
Łącznie dyżurów:    90

--- Analiza Odchyleń ---
Cel I  (Liczba dyżurów = 90):     niedobór = 0, nadmiar = 0
Cel II (Budżet = 45 000 zł):      oszczędność = 0.00 zł, przekroczenie = 0.00 zł
Cel III (Dyżury nocne >= 30):     niedobór = 0, nadmiar = 0

--- Podsumowanie Kosztów ---
Rzeczywisty koszt obsady: 45000.00 zł


### Zadanie 7: Maksymalizacja stopy zwrotu kapitału (ROI)

Inwestor dzieli kapitał między projekt technologiczny oraz rynek nieruchomości:
* Spodziewany roczny zysk wynosi: 0.18 z każdej złotówki zainwestowanej w projekt technologiczny, 0.10 z każdej złotówki w nieruchomościach plus stały roczny dochód pasywny w wysokości 5 tys. zł.
* Łączny koszt obsługi i zabezpieczenia ryzyka wynosi: 0.06 na jednostkę projektu technologicznego, 0.02 na jednostkę nieruchomości oraz 2 tys. zł stałego kosztu rocznego.

Warunki brzegowe:
* Łączny budżet inwestora wynosi maksymalnie 100 tys. zł.
* Zgodnie z polityką bezpieczeństwa, w rynek nieruchomości należy ulokować co najmniej 30 tys. zł.

**Polecenie:**  
Sformułuj zadanie programowania liniowo-ilorazowego maksymalizujące stosunek łącznego zysku do łącznych kosztów ($\text{Zysk} / \text{Koszt}$). Zlinearyzuj model transformacją Charnesa-Coopera i wyznacz optymalną strukturę inwestycji.

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Kolejność: [u1, u2, u0]
# Licznik ułamka po przemnożeniu przez u0: 0.18*u1 + 0.10*u2 + 5*u0
# Ponieważ linprog minimalizuje, odwracamy znaki:
c_frac = np.array([-0.18, -0.10, -5.0])

# Ograniczenie równościowe normalizujące mianownik:
# 0.06*u1 + 0.02*u2 + 2*u0 = 1
A_eq = np.array([[0.06, 0.02, 2.0]])
b_eq = np.array([1.0])

# Ograniczenia nierównościowe przekształcone (wymnożone przez u0):
# 1. x1 + x2 <= 100       => u1 + u2 - 100*u0 <= 0
# 2. x2 >= 30             => -x2 <= -30 => -u2 + 30*u0 <= 0
A_ub = np.array([
    [ 1.0,  1.0, -100.0],
    [ 0.0, -1.0,   30.0]
])
b_ub = np.array([0.0, 0.0])

bounds = [(0, None), (0, None), (0, None)]

result = linprog(
    c=c_frac, 
    A_ub=A_ub, 
    b_ub=b_ub, 
    A_eq=A_eq, 
    b_eq=b_eq, 
    bounds=bounds, 
    method="highs"
)

if result.success:
    u1, u2, u0 = result.x
    
    # Transformacja powrotna do zmiennych pierwotnych:
    x1 = u1 / u0
    x2 = u2 / u0
    max_ratio = -result.fun
    
    zysk = 0.18 * x1 + 0.10 * x2 + 5.0
    koszt = 0.06 * x1 + 0.02 * x2 + 2.0

    print("=" * 50)
    print("        WYNIKI OPTYMALIZACJI (ZADANIE 7)")
    print("=" * 50)
    print(f"Status: {result.message}")
    print(f"Maksymalny zwrot z kapitału (ROI): {max_ratio:.4f}")
    print(f"\nStruktura inwestycji:")
    print(f"  - Projekt technologiczny (x1): {x1:.2f} tys. zł")
    print(f"  - Nieruchomości          (x2): {x2:.2f} tys. zł")
    print(f"  - Łączny kapitał:             {x1 + x2:.2f} tys. zł (limit: <= 100)")
    
    print("\n--- Weryfikacja ---")
    print(f"Roczny zysk:  {zysk:.2f} tys. zł")
    print(f"Roczny koszt: {koszt:.2f} tys. zł")
    print(f"Iloraz zysk/koszt: {zysk / koszt:.4f}")
    print("=" * 50)
else:
    print(f"Błąd optymalizacji: {result.message}")

        WYNIKI OPTYMALIZACJI (ZADANIE 7)
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Maksymalny zwrot z kapitału (ROI): 3.7500

Struktura inwestycji:
  - Projekt technologiczny (x1): 0.00 tys. zł
  - Nieruchomości          (x2): 100.00 tys. zł
  - Łączny kapitał:             100.00 tys. zł (limit: <= 100)

--- Weryfikacja ---
Roczny zysk:  15.00 tys. zł
Roczny koszt: 4.00 tys. zł
Iloraz zysk/koszt: 3.7500


### Zadanie 8: Zysk na jednostkę emisji CO₂

Przedsiębiorstwo realizuje przewozy ładunków dwoma środkami transportu: koleją oraz transportem drogowym (wielkości przewozów mierzone są w tysiącach tonokilometrów – tkm):
* Zysk wynosi: 4 zł za każdy 1 tys. tkm przewieziony koleją, 3 zł za 1 tys. tkm przewieziony drogami oraz 10 zł stałej dopłaty operacyjnej.
* Łączny ślad węglowy wynosi: 1 tona CO₂ na 1 tys. tkm na kolei, 2 tony CO₂ na 1 tys. tkm drogami oraz 4 tony CO₂ stałej emisji bazowej.

Ograniczenia przepustowości:
* Maksymalna przepustowość torów wynosi 40 tys. tkm.
* Maksymalna dostępność floty drogowej wynosi 30 tys. tkm.
* Całkowity wolumen zlecenia musi wynieść co najmniej 25 tys. tkm.

**Polecenie:**  
Znajdź taki podział przewozów między kolej a drogi, który pozwoli zmaksymalizować zysk przypadający na każdą wyemitowaną tonę CO₂ ($\text{Zysk} / \text{Emisja}$).

In [12]:
import numpy as np
from scipy.optimize import linprog

# Kolejność: [u1, u2, u0]
# Licznik do minimalizacji: -(4*u1 + 3*u2 + 10*u0)
c_frac = [-4, -3, -10]

# Normalizacja mianownika (emisja CO2):
# 1*u1 + 2*u2 + 4*u0 = 1
A_eq = [[1, 2, 4]]
b_eq = [1]

# Ograniczenia po pomnożeniu przez u0:
# 1. x1 <= 40        => u1 - 40*u0 <= 0
# 2. x2 <= 30        => u2 - 30*u0 <= 0
# 3. x1 + x2 >= 25   => -u1 - u2 + 25*u0 <= 0
A_ub = [[1, 0, -40],
        [0, 1, -30],
        [-1, -1, 25]
]
b_ub = [0, 0, 0]

bounds = [(0, None)] * 3

result = linprog(
    c=c_frac, 
    A_ub=A_ub, 
    b_ub=b_ub, 
    A_eq=A_eq, 
    b_eq=b_eq, 
    bounds=bounds, 
    method="highs"
)

if result.success:
    u1, u2, u0 = result.x
    x1 = u1 / u0
    x2 = u2 / u0
    max_efficiency = -result.fun
    
    zysk = 4 * x1 + 3 * x2 + 10
    emisja = 1 * x1 + 2 * x2 + 4

    print("=" * 50)
    print("        WYNIKI OPTYMALIZACJI (ZADANIE 8)")
    print("=" * 50)
    print(f"Status: {result.message}")
    print(f"Maksymalna efektywność (Zysk / CO2): {max_efficiency:.4f} zł / tona CO2")
    print(f"\nPodział pracy przewozowej:")
    print(f"  - Transport kolejowy (x1): {x1:.2f} tys. tkm (limit: <= 40)")
    print(f"  - Transport drogowy  (x2): {x2:.2f} tys. tkm (limit: <= 30)")
    print(f"  - Łączny wolumen:          {x1 + x2:.2f} tys. tkm (wymóg: >= 25)")
    
    print("\n--- Weryfikacja ---")
    print(f"Całkowity zysk:   {zysk:.2f} tys. zł")
    print(f"Całkowita emisja: {emisja:.2f} ton CO2")
    print(f"Wskaźnik efektywności: {zysk / emisja:.4f}")
    print("=" * 50)
else:
    print(f"Błąd optymalizacji: {result.message}")

        WYNIKI OPTYMALIZACJI (ZADANIE 8)
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Maksymalna efektywność (Zysk / CO2): 3.8636 zł / tona CO2

Podział pracy przewozowej:
  - Transport kolejowy (x1): 40.00 tys. tkm (limit: <= 40)
  - Transport drogowy  (x2): 0.00 tys. tkm (limit: <= 30)
  - Łączny wolumen:          40.00 tys. tkm (wymóg: >= 25)

--- Weryfikacja ---
Całkowity zysk:   170.00 tys. zł
Całkowita emisja: 44.00 ton CO2
Wskaźnik efektywności: 3.8636


### Zadanie 9: Wydajność pracy na linii produkcyjnej

Zakład montuje dwa warianty podzespołów komputerowych: model podstawowy oraz model gamingowy:
* Łączny wolumen produkcji w sztukach wynosi: 5 sztuk z każdego wariantu podstawowego, 8 sztuk z każdego wariantu gamingowego oraz 20 sztuk z produkcji stałej.
* Łączny nakład pracy wynosi: 2 roboczogodziny na wariant podstawowy, 3 roboczogodziny na wariant gamingowy oraz 5 roboczogodzin stałego czasu przezbrojenia maszyn.

Ograniczenia dostępności zasobów:
* Płyty bazowe: wyprodukowanie wariantu podstawowego wymaga 1 sztuki, a gamingowego 2 sztuk. Dostępny limit wynosi 50 sztuk.
* Czasowochłonność na stanowisku testowym: wariant podstawowy wymaga 3 minut, a gamingowy 1 minuty. Stanowisko dysponuje łącznie 60 minutami.
* Ze względu na minimalne zamówienia klientów należy wyprodukować co najmniej 5 sztuk każdego wariantu.

**Polecenie:**  
Sformułuj i rozwiąż zadanie maksymalizacji wydajności pracy, definiowanej jako liczba zmontowanych sztuk przypadająca na 1 roboczogodzinę ($\text{Sztuki} / \text{Roboczogodziny}$).

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Kolejność: [u1, u2, u0]
# Licznik do minimalizacji: -(5*u1 + 8*u2 + 20*u0)
c_frac = np.array([-5.0, -8.0, -20.0])

# Normalizacja mianownika (roboczogodziny):
# 2*u1 + 3*u2 + 5*u0 = 1
A_eq = np.array([[2.0, 3.0, 5.0]])
b_eq = np.array([1.0])

# Ograniczenia po pomnożeniu przez u0:
# 1. Płyty: x1 + 2*x2 <= 50        => u1 + 2*u2 - 50*u0 <= 0
# 2. Czas testów: 3*x1 + x2 <= 60  => 3*u1 + u2 - 60*u0 <= 0
# 3. Min. produkcja x1 >= 5        => -u1 + 5*u0 <= 0
# 4. Min. produkcja x2 >= 5        => -u2 + 5*u0 <= 0
A_ub = np.array([
    [ 1.0,  2.0, -50.0],
    [ 3.0,  1.0, -60.0],
    [-1.0,  0.0,   5.0],
    [ 0.0, -1.0,   5.0]
])
b_ub = np.array([0.0, 0.0, 0.0, 0.0])

bounds = [(0, None), (0, None), (0, None)]

result = linprog(
    c=c_frac, 
    A_ub=A_ub, 
    b_ub=b_ub, 
    A_eq=A_eq, 
    b_eq=b_eq, 
    bounds=bounds, 
    method="highs"
)

if result.success:
    u1, u2, u0 = result.x
    x1 = u1 / u0
    x2 = u2 / u0
    max_productivity = -result.fun
    
    wyprodukowane_sztuki = 5 * x1 + 8 * x2 + 20
    naklad_pracy = 2 * x1 + 3 * x2 + 5

    print("=" * 50)
    print("        WYNIKI OPTYMALIZACJI (ZADANIE 9)")
    print("=" * 50)
    print(f"Status: {result.message}")
    print(f"Maksymalna wydajność pracy: {max_productivity:.4f} sztuk / roboczogodzinę")
    print(f"\nOptymalna wielkość partii:")
    print(f"  - Model standardowy (x1): {x1:.2f} szt.")
    print(f"  - Model gamingowy   (x2): {x2:.2f} szt.")
    
    print("\n--- Wykorzystanie Zasobów ---")
    print(f"Płyty bazowe (<=50):     {1*x1 + 2*x2:.2f} szt.")
    print(f"Czas testowania (<=60):  {3*x1 + 1*x2:.2f} min")
    
    print("\n--- Weryfikacja Wydajności ---")
    print(f"Łączna produkcja: {wyprodukowane_sztuki:.2f} sztuk")
    print(f"Łączny czas:      {naklad_pracy:.2f} roboczogodzin")
    print(f"Wskaźnik:         {wyprodukowane_sztuki / naklad_pracy:.4f} szt./rbh")
    print("=" * 50)
else:
    print(f"Błąd optymalizacji: {result.message}")

        WYNIKI OPTYMALIZACJI (ZADANIE 9)
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Maksymalna wydajność pracy: 2.8333 sztuk / roboczogodzinę

Optymalna wielkość partii:
  - Model standardowy (x1): 5.00 szt.
  - Model gamingowy   (x2): 5.00 szt.

--- Wykorzystanie Zasobów ---
Płyty bazowe (<=50):     15.00 szt.
Czas testowania (<=60):  20.00 min

--- Weryfikacja Wydajności ---
Łączna produkcja: 85.00 sztuk
Łączny czas:      30.00 roboczogodzin
Wskaźnik:         2.8333 szt./rbh
